<a href="https://colab.research.google.com/github/garykbrixi/minerva/blob/main/examples/notebooks/rna_structure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNA secondary structure with Minerva

Minerva's `base_pairing` head returns a dense contact map. This notebook calls a
discrete structure from it, draws it, and adds an interactive viewer with a live
threshold slider.

The viewer loads [fornac](https://github.com/ViennaRNA/fornac) from a CDN at
runtime, so nothing is bundled into the `minerva` package. It needs network
access from the output frame, which Colab allows.

In [ ]:
import importlib.util
if importlib.util.find_spec("minerva") is None:
    !pip install -q "minerva-dna[viz] @ git+https://github.com/garykbrixi/minerva.git"

## Call a structure

Passing `tokens` crops the contact map to nucleotide positions, dropping the
`<+>` marker and any amino-acid stretch.

In [ ]:
import torch
from transformers import AutoTokenizer
from minerva import MinervaForMaskedLM
from minerva.rna_structure import call_structure

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MinervaForMaskedLM.from_pretrained("gbrixi/minerva-mlm").to(device).eval()
tokenizer = AutoTokenizer.from_pretrained("gbrixi/minerva-mlm")

sequence = "<+>cgcggggtggagcagcctggtagctcgtcgggctcataacccgaagatcgtcggttcaaatccggcccccgcaacca"
tokens = tokenizer(sequence, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model(**tokens, output_interactions=True)

token_list = tokenizer.convert_ids_to_tokens(tokens["input_ids"][0].tolist())
s = call_structure(out.interactions["base_pairing"], tokens=token_list, name="tRNA")

print(s)
print(s.sequence)
print(s.dot_bracket)

## Static drawing

Geometry comes from ViennaRNA's naview layout applied to *this* structure —
nothing is folded, so the picture is always Minerva's prediction.

In [ ]:
s.plot(figsize=(8, 8));

## Interactive viewer

`at_threshold` re-calls the structure from the stored candidates, so the slider
needs no contact map and no server. The readout reports pairs that are too
knotted to write in dot-bracket notation — those are absent from the drawing but
still present in `s.pairs` and `s.to_ct()`.

In [ ]:
import html, json
from IPython.display import HTML, display
from ipywidgets import interact, FloatSlider

FORNAC = "https://cdn.jsdelivr.net/npm/fornac@1.1.8/dist/scripts/fornac.js"

def forna(structure, height=420):
    """Render a structure with fornac, in a sandboxed iframe."""
    page = f"""<!doctype html><meta charset="utf-8">
<body style="margin:0;font-family:Helvetica,Arial,sans-serif">
<div id="c" style="width:100%;height:{height}px"></div>
<script src="{FORNAC}"></script>
<script>
  new fornac.FornaContainer("#c", {{applyForce:true, allowPanningAndZooming:true}})
      .addRNA({json.dumps(structure.dot_bracket)},
              {{sequence:{json.dumps(structure.sequence)}, name:{json.dumps(structure.name)}}});
</script></body>"""
    return HTML(f'<iframe srcdoc="{html.escape(page, quote=True)}" '
                f'style="width:100%;height:{height + 10}px;border:1px solid #e0e0e0" '
                'sandbox="allow-scripts"></iframe>')

@interact(threshold=FloatSlider(value=0.6, min=0.1, max=0.95, step=0.05,
                                continuous_update=False, description="threshold"))
def view(threshold):
    t = s.at_threshold(threshold)
    note = f"{len(t.pairs)} pairs"
    if t.pseudoknot_pairs:
        note += f" \u00b7 {len(t.pseudoknot_pairs)} pseudoknotted"
    if t.dropped_pairs:
        note += f" \u00b7 {len(t.dropped_pairs)} too knotted to notate (not drawn)"
    print(note)
    display(forna(t))

## Exports

The dot-bracket string is the portable artifact — sequence plus structure is what
RNAfold, [forna](http://rna.tbi.univie.ac.at/forna/), VARNA and R2R all read.
The connect table keeps every pair, including pseudoknots that dot-bracket
cannot express.

In [ ]:
s.to_vienna("structure.fa")
s.to_ct("structure.ct")
print(open("structure.fa").read())

## Eukaryotic RNA

Minerva-MLM is trained on prokaryotic genomes. For eukaryotic RNA there is a RiNALMo-based
checkpoint with the same `base_pairing` head, so `call_structure` works unchanged. It takes plain
nucleotides with no `<+>` marker. See
[eukaryotic RNA support](https://github.com/garykbrixi/minerva/tree/main/examples/eukaryotic_rna).

In [ ]:
from minerva.modeling_rinalmo import RiNALMoMinervaForMaskedLM

repo = "gbrixi/minerva-rinalmo-giga"
rinalmo = RiNALMoMinervaForMaskedLM.from_pretrained(repo).to(device).eval()
rna_tokenizer = AutoTokenizer.from_pretrained(repo)

rna = "GGGGCUUUAGCUCAGCUGGGAGAGCGCCUGCCUUGCACGCAGGAGGUCAGCGGUUCGAUCCGCUAAGCUCCA"
with torch.no_grad():
    out = rinalmo(**rna_tokenizer(rna, return_tensors="pt").to(device), output_interactions=True)

# This map already excludes the boundary tokens, so pass the sequence instead of tokens.
e = call_structure(out.interactions["base_pairing"], rna, name="tRNA (RiNALMo)")
print(e.dot_bracket)
e.plot(figsize=(8, 8));